# Escalón 3 · Qwen3-0.6B-Base + LoRA con salida restringida

LoRA sobre `Qwen/Qwen3-0.6B-Base` (Apache-2.0) con Unsloth en GPU. La decodificación pasa por `CodeTrie`: el modelo sólo puede devolver un código del catálogo de entrenamiento o «no sé» (prueba T-05, cubierta por `tests/unit/test_constrained.py`).

La búsqueda en haz por ítem es lenta, así que validación y prueba se evalúan en una **muestra aleatoria fija** (2 000 y 5 000 descripciones). Se reporta así, sin extrapolar.

In [ ]:
# Entorno: el repositorio público y sus dependencias de GPU (torch ya viene en Kaggle).
!git clone --depth 1 https://github.com/DiegoFranciscoG/cpc-mini.git
%cd cpc-mini
!pip install -q -e ".[gpu]"
!pip install -q unsloth

In [ ]:
# Datos: sube data/processed (train, val, test, catalog y manifest) como dataset PRIVADO de Kaggle
# llamado "cpc-mini-processed". El cuaderno no descarga nada del SERCOP.
import json, time
from pathlib import Path
from cpcmini.config import load_settings
from cpcmini.data.dataset import read_manifest, read_split
from cpcmini.pipeline import cap_per_class, evaluate_model, write_report

PROCESSED = "/kaggle/input/cpc-mini-processed"
OUT = Path("/kaggle/working")
settings = load_settings("configs")
manifest = read_manifest(PROCESSED)
print("dataset", manifest["dataset_hash"][:12])
train = cap_per_class(read_split(PROCESSED, "train"), settings.models.linear.max_per_class)
print(train.height, "descripciones de entrenamiento")

In [ ]:
# Muestra fija de validación y prueba (mismo protocolo, menos ítems).
import shutil
SAMPLE = OUT / "processed_sample"
SAMPLE.mkdir(exist_ok=True)
for name, n in (("val", 2000), ("test", 5000)):
    frame = read_split(PROCESSED, name)
    frame.sample(n=min(n, frame.height), seed=settings.models.seed).write_parquet(SAMPLE / f"{name}.parquet")
for name in ("train.parquet", "catalog.parquet", "manifest.json"):
    shutil.copy(Path(PROCESSED) / name, SAMPLE / name)

In [ ]:
# Entrenamiento LoRA (Unsloth si hay GPU; PEFT estándar si no).
from cpcmini.models.generative_hf import GenerativeClassifier, train_lora

start = time.perf_counter()
folder = train_lora(train["desc_norm"].to_list(), train["code9"].to_list(),
                    settings.models.generative, OUT / "qwen3_lora", settings.models.seed)
train_seconds = time.perf_counter() - start
print(f"{train_seconds/60:.1f} min")

In [ ]:
# Evaluación con decodificación restringida al catálogo de entrenamiento.
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(folder / "adapter")
base = AutoPeftModelForCausalLM.from_pretrained(folder / "adapter").to("cuda" if torch.cuda.is_available() else "cpu")
base.eval()
catalog = sorted(set(train["code9"].to_list()))
model = GenerativeClassifier(base, tokenizer, settings.models.generative, catalog,
                             abstain_label=settings.eval.abstain_label)
metrics, temperature, threshold = evaluate_model(model, "qwen3_lora", settings, SAMPLE,
                                                 train.height, train_seconds)
metrics["evaluated_on"] = "muestra fija: 2 000 de validación y 5 000 de prueba"
write_report(metrics, OUT / "reports")
print(json.dumps({k: metrics["test"][k]["top1"] for k in ("level_5", "level_9")}, indent=1))

## Exportación GGUF (sólo si gana)

Con Unsloth: `model.save_pretrained_gguf("qwen3_cpc", tokenizer, quantization_method="q4_k_m")` y prueba con llama.cpp en CPU. T-07: la versión cuantizada no puede perder más de 2 puntos de top-1 frente a la original.